In [7]:
print("start!!!!!!!!!!!")

start!!!!!!!!!!!


In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import zscore





In [9]:
df= pd.read_csv("DATA/DataCoSupplyChainDataset.csv",  encoding="latin-1")

df.shape

(180519, 53)

In [10]:
df.head(3)

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class


In [11]:
print("Dataset column: ", df.columns.tolist())
print("Data types: ", df.dtypes)
print("Missing values: ", df.isnull().sum())
print("Summary statistics: ", df.describe())

Dataset column:  ['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Delivery Status', 'Late_delivery_risk', 'Category Id', 'Category Name', 'Customer City', 'Customer Country', 'Customer Email', 'Customer Fname', 'Customer Id', 'Customer Lname', 'Customer Password', 'Customer Segment', 'Customer State', 'Customer Street', 'Customer Zipcode', 'Department Id', 'Department Name', 'Latitude', 'Longitude', 'Market', 'Order City', 'Order Country', 'Order Customer Id', 'order date (DateOrders)', 'Order Id', 'Order Item Cardprod Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Order Region', 'Order State', 'Order Status', 'Order Zipcode', 'Product Card Id', 'Product Category Id', 'Product Description', 'Product Image', 'Product Name', 'Product Price', 'Product Status', 'shippi

In [13]:
columns = [

    "Shipping Mode",
    "Delivery Status",
    "Late_delivery_risk",
    "Days for shipping (real)",
    "Days for shipment (scheduled)",
    "Order Profit Per Order",
    "Sales",
    "Order Item Discount Rate",
    "Customer Segment",
    "Market",
    "Category Name"
]

dataset = df[columns].copy()

path= "DATA/finalDataset.csv"
dataset.to_csv(path, index=False)

dataset.shape

(180519, 11)

In [14]:
dataset.head(4)

,Shipping Mode,Delivery Status,Late_delivery_risk,Days for shipping (real),Days for shipment (scheduled),Order Profit Per Order,Sales,Order Item Discount Rate,Customer Segment,Market,Category Name
0,Standard Class,Advance shipping,0,3,4,91.250000,327.75,0.04,Consumer,Pacific Asia,Sporting Goods
1,Standard Class,Late delivery,1,5,4,-249.089996,327.75,0.05,Consumer,Pacific Asia,Sporting Goods
2,Standard Class,Shipping on time,0,4,4,-247.779999,327.75,0.06,Consumer,Pacific Asia,Sporting Goods
3,Standard Class,Advance shipping,0,3,4,22.860001,327.75,0.07,Home Office,Pacific Asia,Sporting Goods


In [15]:
print("Dataset column: ", dataset.columns.tolist())
print("Data types: ", dataset.dtypes)
print("Missing values: ", dataset.isnull().sum())
print("Summary statistics: ", dataset.describe())

Dataset column:  ['Shipping Mode', 'Delivery Status', 'Late_delivery_risk', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Order Profit Per Order', 'Sales', 'Order Item Discount Rate', 'Customer Segment', 'Market', 'Category Name']
Data types:  Shipping Mode                        str
Delivery Status                      str
Late_delivery_risk                 int64
Days for shipping (real)           int64
Days for shipment (scheduled)      int64
Order Profit Per Order           float64
Sales                            float64
Order Item Discount Rate         float64
Customer Segment                     str
Market                               str
Category Name                        str
dtype: object
Missing values:  Shipping Mode                    0
Delivery Status                  0
Late_delivery_risk               0
Days for shipping (real)         0
Days for shipment (scheduled)    0
Order Profit Per Order           0
Sales                            0
Order Item Di

DATA CLEANING STEPS: Duplicate datas,invalids, outliers, stripping whitespace,type conversion 

In [16]:

duplicates = dataset.duplicated().sum()
print("Number of duplicate rows: ", duplicates)

Number of duplicate rows:  6364


In [17]:
if duplicates > 0:
    dataset = dataset.drop_duplicates()
    print("Duplicate rows removed. New shape: ", dataset.shape)

Duplicate rows removed. New shape:  (174155, 11)


In [18]:
invalid_dimensions_mask = (dataset['Late_delivery_risk'] <= 0) | (dataset['Days for shipping (real)'] <= 0) | (dataset['Days for shipment (scheduled)'] <= 0) | (dataset['Order Profit Per Order'] <= 0) | (dataset['Sales'] <= 0)| (dataset['Order Item Discount Rate'] <= 0)
invalid_count = invalid_dimensions_mask.sum()
print("Number of rows with invalid dimensions: ", invalid_count)

Number of rows with invalid dimensions:  105367


In [19]:
if invalid_count > 0:
    dataset = dataset[~invalid_dimensions_mask]
    print("Invalid dimension rows removed. New shape: ", dataset.shape)

Invalid dimension rows removed. New shape:  (68788, 11)


In [20]:
Cols=['Days for shipping (real)',
    'Days for shipment (scheduled)',
    'Order Profit Per Order',
    'Sales',
    'Order Item Discount Rate']
threshold = 2
for c in Cols:
    z_scores= zscore(dataset[c])
    outliers = np.where(np.abs(z_scores) > threshold) 
    
    print(f"Outliers in {c} : {len(outliers[0])} ")


Outliers in Days for shipping (real) : 0 
Outliers in Days for shipment (scheduled) : 0 
Outliers in Order Profit Per Order : 2636 
Outliers in Sales : 1388 
Outliers in Order Item Discount Rate : 4112 


In [21]:

capping= dataset.copy()

z= zscore(dataset['Order Profit Per Order'])
capping['Order Profit Per Order'] = np.where( z > threshold,
                            dataset['Order Profit Per Order'].mean() + threshold * dataset['Order Profit Per Order'].std(),
                            dataset['Order Profit Per Order'])
capping['Order Profit Per Order'] = np.where(z < -threshold,
                            dataset['Order Profit Per Order'].mean() - threshold * dataset['Order Profit Per Order'].std(),
                            capping['Order Profit Per Order'])



z= zscore(dataset['Sales'])
capping['Sales'] = np.where( z > threshold,
                            dataset['Sales'].mean() + threshold * dataset['Sales'].std(),
                            dataset['Sales'])
capping['Sales'] = np.where(z < -threshold,
                            dataset['Sales'].mean() - threshold * dataset['Sales'].std(),
                            capping['Sales'])


z= zscore(dataset['Order Item Discount Rate'])
capping['Order Item Discount Rate'] = np.where( z > threshold,
                            dataset['Order Item Discount Rate'].mean() + threshold * dataset['Order Item Discount Rate'].std(),
                            dataset['Order Item Discount Rate'])
capping['Order Item Discount Rate'] = np.where(z < -threshold,
                            dataset['Order Item Discount Rate'].mean() - threshold * dataset['Order Item Discount Rate'].std(),
                            capping['Order Item Discount Rate'])


print("Original shape  :", dataset.shape)
print("Capped shape    :", capping.shape)

dataset= capping
dataset.shape


Original shape  : (68788, 11)
Capped shape    : (68788, 11)


(68788, 11)

In [23]:
cols = ['Shipping Mode', 'Delivery Status', 'Customer Segment', 'Market', 'Category Name']
for col in cols:
    unique_vals = dataset[col].unique()
    print(f"\n[{col}]  has   {len(unique_vals)} unique values")
    for val in sorted(unique_vals):
        count = (dataset[col] == val).sum()
        print(f"  {repr(val):<35}  {count:>6,}")


[Shipping Mode]  has   3 unique values
  'First Class'                        19,038
  'Second Class'                       20,264
  'Standard Class'                     29,486

[Delivery Status]  has   1 unique values
  'Late delivery'                      68,788

[Customer Segment]  has   3 unique values
  'Consumer'                           35,083
  'Corporate'                          21,018
  'Home Office'                        12,687

[Market]  has   5 unique values
  'Africa'                              4,527
  'Europe'                             19,068
  'LATAM'                              19,239
  'Pacific Asia'                       15,965
  'USCA'                                9,989

[Category Name]  has   50 unique values
  'Accessories'                           740
  'As Seen on  TV!'                        26
  'Baby '                                  73
  'Baseball & Softball'                   264
  'Basketball'                             29
  'Books '         

In [24]:
dataset.drop(columns=['Delivery Status'], inplace= True)
dataset.shape

(68788, 10)

In [29]:
dataset['Category Name'] = dataset['Category Name'].str.strip()
print(dataset['Category Name'].unique())


<StringArray>
[      'Sporting Goods',      'Women's Apparel',         'Boxing & MMA',
               'Cleats',        'Shop By Sport',          'Electronics',
     'Cardio Equipment',             'Trade-In',       'Men's Footwear',
     'Camping & Hiking',              'Cameras', 'Consumer Electronics',
            'Computers',           'Basketball',               'Soccer',
  'Baseball & Softball',     'Women's Clothing',          'Accessories',
  'Fitness Accessories',      'As Seen on  TV!',       'Girls' Apparel',
           'Golf Balls',     'Tennis & Racquet',    'Strength Training',
               'Crafts',  'Children's Clothing',       'Men's Clothing',
                 'Baby',                'Books',     'Kids' Golf Clubs',
              'Fishing',                 'DVDs',                  'CDs',
   'Hunting & Shooting',               'Garden',               'Hockey',
         'Pet Supplies',    'Health and Beauty',                'Music',
          'Video Games',             

In [30]:
for col in cols:
    dataset[col] = pd.Categorical(dataset[col])

print(dataset[cols].dtypes)

Shipping Mode       category
Customer Segment    category
Market              category
Category Name       category
dtype: object


In [31]:
dataset['Late_delivery_risk'] = dataset['Late_delivery_risk'].astype(bool)


print(dataset['Late_delivery_risk'].dtype)
print(dataset['Late_delivery_risk'].value_counts())

bool
Late_delivery_risk
True    68788
Name: count, dtype: int64


In [33]:
print(dataset.dtypes)
dataset.shape

Shipping Mode                    category
Late_delivery_risk                   bool
Days for shipping (real)            int64
Days for shipment (scheduled)       int64
Order Profit Per Order            float64
Sales                             float64
Order Item Discount Rate          float64
Customer Segment                 category
Market                           category
Category Name                    category
dtype: object


(68788, 10)

Feature Engineering: Shipping Delay, Profit Margin, Ordered Categorical[Shipping Mode] , Shipping Speed Score

In [35]:

dataset['Shipping Delay'] = dataset['Days for shipping (real)'] - dataset['Days for shipment (scheduled)']

dataset['Profit Margin'] = (dataset['Order Profit Per Order'] / dataset['Sales']) * 100


print(dataset[['Shipping Delay', 'Profit Margin']].describe().round(2))
dataset.head(5)

       Shipping Delay  Profit Margin
count        68788.00       68788.00
mean             1.66          25.89
std              0.90          12.46
min              1.00           0.71
25%              1.00          15.95
50%              1.00          27.23
75%              2.00          35.34
max              4.00          49.52


,Shipping Mode,Late_delivery_risk,Days for shipping (real),Days for shipment (scheduled),Order Profit Per Order,Sales,Order Item Discount Rate,Customer Segment,Market,Category Name,Shipping Delay,Profit Margin
6,First Class,True,2,1,95.180000,327.75,0.12,Home Office,Pacific Asia,Sporting Goods,1,29.040427
7,First Class,True,2,1,68.430000,327.75,0.13,Corporate,Pacific Asia,Sporting Goods,1,20.878719
8,Second Class,True,3,2,133.720001,327.75,0.15,Corporate,Pacific Asia,Sporting Goods,1,40.799390
9,First Class,True,2,1,132.149994,327.75,0.16,Corporate,Pacific Asia,Sporting Goods,1,40.320364
11,Second Class,True,5,2,45.689999,327.75,0.18,Consumer,Pacific Asia,Sporting Goods,3,13.940503


In [36]:
speed_order = ['Standard Class', 'Second Class', 'First Class']
dataset['Shipping Mode'] = pd.Categorical(dataset['Shipping Mode'], 
                                           categories=speed_order, 
                                           ordered=True)

print(dataset['Shipping Mode'].cat.categories.tolist())

dataset['Shipping Speed Score'] = dataset['Shipping Mode'].cat.codes + 1
print(dataset.groupby('Shipping Mode')['Shipping Speed Score'].first().reset_index())


['Standard Class', 'Second Class', 'First Class']
    Shipping Mode  Shipping Speed Score
0  Standard Class                     1
1    Second Class                     2
2     First Class                     3
